<a href="https://colab.research.google.com/github/TAUforPython/Graph-MachineLearning/blob/main/utils_ERD2MMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ERD to Mermaid Converter
# Run this cell to install any necessary dependencies (if needed)
!pip install lxml beautifulsoup4

In [2]:
import re
import xml.etree.ElementTree as ET
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os

In [4]:


def parse_erd_file(file_content):
    """Parse the ERD XML file and extract entities and relationships"""
    try:
        # Parse XML
        root = ET.fromstring(file_content)

        # Extract entities
        entities = {}
        data_source = root.find('.//data-source')
        if data_source is not None:
            for entity in data_source.findall('entity'):
                entity_id = entity.get('id')
                entity_name = entity.get('name')
                entities[entity_id] = {
                    'name': entity_name,
                    'fields': []  # We'll populate this from relationships
                }

        # Extract relationships
        relations = []
        relations_elem = root.find('relations')
        if relations_elem is not None:
            for relation in relations_elem.findall('relation'):
                rel_name = relation.get('name', '')
                rel_type = relation.get('type', '')
                pk_ref = relation.get('pk-ref', '')
                fk_ref = relation.get('fk-ref', '')

                if pk_ref in entities and fk_ref in entities:
                    relations.append({
                        'name': rel_name,
                        'type': rel_type,
                        'pk_entity': entities[pk_ref]['name'],
                        'fk_entity': entities[fk_ref]['name'],
                        'pk_id': pk_ref,
                        'fk_id': fk_ref
                    })

        return entities, relations

    except ET.ParseError as e:
        print(f"Error parsing XML: {e}")
        return None, None

def determine_cardinality(relation_name):
    """Determine cardinality based on relation name patterns"""
    relation_lower = relation_name.lower()

    # Check for many-to-many indicators
    if 'link' in relation_lower or 'connection' in relation_lower:
        return '}|--|{'

    # Check for one-to-many indicators (most common)
    if '_fk' in relation_lower or 'fk_' in relation_lower:
        return '||--o{'

    # Default to one-to-many from PK to FK
    return '||--o{'

def generate_mermaid_erd(entities, relations, max_tables=30):
    """Generate Mermaid ERD format from parsed data"""

    mermaid_lines = ["erDiagram"]

    # Track which entities are used in relationships
    used_entities = set()

    # Process relationships
    for rel in relations[:max_tables * 2]:  # Limit relationships to keep diagram manageable
        pk_entity = rel['pk_entity']
        fk_entity = rel['fk_entity']
        used_entities.add(pk_entity)
        used_entities.add(fk_entity)

        cardinality = determine_cardinality(rel['name'])

        # Add relationship line
        mermaid_lines.append(f"    {pk_entity} {cardinality} {fk_entity} : {rel.get('name', 'relates_to')}")

    # Add entity definitions for used entities (limited to max_tables)
    entities_added = 0
    for entity_id, entity_info in entities.items():
        if entity_info['name'] in used_entities and entities_added < max_tables:
            mermaid_lines.append(f"    {entity_info['name']} {{")
            mermaid_lines.append(f"        int Code PK")
            mermaid_lines.append(f"        string Name")
            mermaid_lines.append(f"    }}")
            entities_added += 1

    return "\n".join(mermaid_lines)

def save_mermaid_file(content, filename="database_erd.mmd"):
    """Save Mermaid content to a file"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)
    return filename

def parse_entity_fields_from_relations(relations):
    """Try to infer fields from relationship patterns"""
    field_patterns = {}

    for rel in relations:
        # Extract potential field names from relationship names
        rel_name = rel.get('name', '')

        # Look for common field patterns
        id_match = re.search(r'([A-Za-z_]+)(?:Id|_code|_id|_Code|_ID)', rel_name)
        if id_match:
            field_name = id_match.group(1)
            entity_name = rel.get('fk_entity', '')
            if entity_name and entity_name not in field_patterns:
                if field_name not in field_patterns.get(entity_name, []):
                    field_patterns.setdefault(entity_name, set()).add(field_name)

    return field_patterns

In [5]:
# @title Upload and Convert ERD File
# Create upload button
upload_button = widgets.FileUpload(
    accept='.txt, .xml',
    multiple=False,
    description='Upload ERD file:'
)

output = widgets.Output()
max_tables_slider = widgets.IntSlider(
    value=50,
    min=5,
    max=1000,
    step=5,
    description='Max tables:',
    style={'description_width': 'initial'}
)

convert_button = widgets.Button(
    description='Convert to Mermaid',
    button_style='primary'
)

download_button = widgets.Button(
    description='Download Mermaid File',
    button_style='success',
    disabled=True
)

display(upload_button, max_tables_slider, convert_button, download_button, output)

# Global variable to store generated content
generated_content = ""
mermaid_filename = ""

def on_convert_clicked(b):
    global generated_content, mermaid_filename
    with output:
        clear_output(wait=True)

        if not upload_button.value:
            print("Please upload a file first.")
            return

        # Get uploaded file content
        uploaded_file = list(upload_button.value.values())[0]
        file_content = uploaded_file['content'].decode('utf-8')

        print("Processing ERD file...")

        # Parse the ERD
        entities, relations = parse_erd_file(file_content)

        if entities and relations:
            print(f"Found {len(entities)} entities and {len(relations)} relationships")
            print(f"Generating Mermaid diagram with max {max_tables_slider.value} tables...")

            # Generate Mermaid ERD
            mermaid_content = generate_mermaid_erd(entities, relations, max_tables_slider.value)

            # Save to file
            mermaid_filename = save_mermaid_file(mermaid_content)
            generated_content = mermaid_content

            # Display preview
            print("\n" + "="*50)
            print("MERMAID ERD PREVIEW:")
            print("="*50)
            print(mermaid_content)
            print("="*50)
            print(f"\n✅ Conversion complete! File saved as: {mermaid_filename}")

            # Enable download button
            download_button.disabled = False
        else:
            print("Failed to parse ERD file. Please check the file format.")

def on_download_clicked(b):
    if generated_content and mermaid_filename:
        with open(mermaid_filename, 'w', encoding='utf-8') as f:
            f.write(generated_content)
        files.download(mermaid_filename)

convert_button.on_click(on_convert_clicked)
download_button.on_click(on_download_clicked)
'''
# @title Alternative: Paste XML Content Directly
xml_input = widgets.Textarea(
    value='',
    placeholder='Paste your XML/ERD content here...',
    description='XML Content:',
    layout=widgets.Layout(width='100%', height='200px')
)

convert_xml_button = widgets.Button(
    description='Convert Pasted XML',
    button_style='warning'
)

xml_output = widgets.Output()

display(xml_input, convert_xml_button, xml_output)

def on_convert_xml_clicked(b):
    global generated_content, mermaid_filename
    with xml_output:
        clear_output(wait=True)

        if not xml_input.value:
            print("Please paste XML content first.")
            return

        print("Processing pasted XML content...")

        # Parse the ERD
        entities, relations = parse_erd_file(xml_input.value)

        if entities and relations:
            print(f"Found {len(entities)} entities and {len(relations)} relationships")

            # Generate Mermaid ERD
            mermaid_content = generate_mermaid_erd(entities, relations, 25)

            # Save to file
            mermaid_filename = save_mermaid_file(mermaid_content, "pasted_erd.mmd")
            generated_content = mermaid_content

            # Display preview
            print("\n" + "="*50)
            print("MERMAID ERD PREVIEW:")
            print("="*50)
            print(mermaid_content[:500] + "..." if len(mermaid_content) > 500 else mermaid_content)
            print("="*50)
            print(f"\n✅ Conversion complete! File saved as: {mermaid_filename}")

            # Create download button for this output
            download_pasted = widgets.Button(description='Download Pasted Result')
            display(download_pasted)

            def download_pasted_file(b):
                files.download(mermaid_filename)

            download_pasted.on_click(download_pasted_file)
        else:
            print("Failed to parse XML content. Please check the format.")

convert_xml_button.on_click(convert_xml_clicked)
'''
# @title View Relationship Statistics
if 'entities' in locals() and 'relations' in locals():
    stats_button = widgets.Button(description='Show Statistics')
    stats_output = widgets.Output()

    display(stats_button, stats_output)

    def show_stats(b):
        with stats_output:
            clear_output(wait=True)
            print("📊 RELATIONSHIP STATISTICS")
            print("="*40)

            # Count relationships by entity
            entity_relations = {}
            for rel in relations[:100]:  # Limit to first 100 for performance
                pk = rel['pk_entity']
                fk = rel['fk_entity']
                entity_relations[pk] = entity_relations.get(pk, 0) + 1
                entity_relations[fk] = entity_relations.get(fk, 0) + 1

            # Sort by number of relationships
            sorted_entities = sorted(entity_relations.items(), key=lambda x: x[1], reverse=True)

            print("Top 10 entities by relationship count:")
            for entity, count in sorted_entities[:10]:
                print(f"  • {entity}: {count} relationships")

            print(f"\nTotal unique entities with relationships: {len(entity_relations)}")

            # Show relationship type distribution
            rel_types = {}
            for rel in relations[:100]:
                rel_type = rel.get('type', 'unknown')
                rel_types[rel_type] = rel_types.get(rel_type, 0) + 1

            print("\nRelationship types:")
            for rel_type, count in rel_types.items():
                print(f"  • {rel_type}: {count}")

    stats_button.on_click(show_stats)

print("\n📝 Instructions:")
print("1. Click 'Choose File' to upload your DB_MIS_REPORT.txt file")
print("2. Adjust the 'Max tables' slider to control diagram size")
print("3. Click 'Convert to Mermaid' to generate the diagram")
print("4. Click 'Download Mermaid File' to save the .mmd file")
print("\nYou can also paste XML content directly in the second section.")

FileUpload(value={}, accept='.txt, .xml', description='Upload ERD file:')

IntSlider(value=50, description='Max tables:', max=1000, min=5, step=5, style=SliderStyle(description_width='i…

Button(button_style='primary', description='Convert to Mermaid', style=ButtonStyle())

Button(button_style='success', description='Download Mermaid File', disabled=True, style=ButtonStyle())

Output()


📝 Instructions:
1. Click 'Choose File' to upload your DB_MIS_REPORT.txt file
2. Adjust the 'Max tables' slider to control diagram size
3. Click 'Convert to Mermaid' to generate the diagram
4. Click 'Download Mermaid File' to save the .mmd file

You can also paste XML content directly in the second section.
